In [2]:
from google.colab import files
uploaded = files.upload()

Saving TelcoCustomerChurnDataset.csv to TelcoCustomerChurnDataset.csv


In [3]:
# Data Cleaning and Preprocessing

import pandas as pd

# Load dataset
df = pd.read_csv("TelcoCustomerChurnDataset.csv")

# Check first 5 rows
print(df.head())

# Convert TotalCharges to numeric
# Some values are blank spaces, so errors='coerce' converts them to NaN

df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

# Fill missing values with median

df['TotalCharges'] = df['TotalCharges'].fillna(
    df['TotalCharges'].median()
)

# Remove unnecessary column

df.drop('customerID', axis=1, inplace=True)

# Remove duplicate rows

df = df.drop_duplicates()

# Final check

print(df.info())

print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [4]:
# ============================================
# FEATURE ENGINEERING
# ============================================

# 1. Average Monthly Spending
# Total amount paid divided by tenure

df['AvgMonthlySpend'] = (
    df['TotalCharges'] / (df['tenure'] + 1)
)

# 2. Tenure Group
# Categorize customers based on duration

df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=[
        '0-1 Year',
        '1-2 Years',
        '2-4 Years',
        '4-6 Years'
    ]
)

# 3. Number of Services Used

services = [
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

# Count active services

df['TotalServices'] = (
    (df[services] != 'No').sum(axis=1)
)

# Display updated dataset

print(df.head())

   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  ...  \
0  No phone service             DSL             No          Yes  ...   
1                No             DSL            Yes           No  ...   
2                No             DSL            Yes          Yes  ...   
3  No phone service             DSL            Yes           No  ...   
4                No     Fiber optic             No           No  ...   

  StreamingMovies        Contract PaperlessBilling              PaymentMethod  \
0              No  Month-to-month              Yes           Electronic check

In [5]:

# ENCODE CATEGORICAL VARIABLES

# Convert categorical columns into numerical columns

df = pd.get_dummies(
    df,
    drop_first=True
)

# Display dataset shape
print(df.shape)

# Display first 5 rows
print(df.head())

(7021, 36)
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  AvgMonthlySpend  \
0              0       1           29.85         29.85        14.925000   
1              0      34           56.95       1889.50        53.985714   
2              0       2           53.85        108.15        36.050000   
3              0      45           42.30       1840.75        40.016304   
4              0       2           70.70        151.65        50.550000   

   TotalServices  gender_Male  Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0              3        False         True           False             False   
1              4         True        False           False              True   
2              4         True        False           False              True   
3              5         True        False           False             False   
4              2        False        False           False              True   

   ...  Contract_One year  Contract_Two year  PaperlessBi

In [6]:
#SCALING
# define features and target
# Features
X = df.drop('Churn_Yes', axis=1)

# Target
y = df['Churn_Yes']

print(X.head())

print(y.head())

   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  AvgMonthlySpend  \
0              0       1           29.85         29.85        14.925000   
1              0      34           56.95       1889.50        53.985714   
2              0       2           53.85        108.15        36.050000   
3              0      45           42.30       1840.75        40.016304   
4              0       2           70.70        151.65        50.550000   

   TotalServices  gender_Male  Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0              3        False         True           False             False   
1              4         True        False           False              True   
2              4         True        False           False              True   
3              5         True        False           False             False   
4              2        False        False           False              True   

   ...  StreamingMovies_Yes  Contract_One year  Contract_Two year  \

In [7]:
#split dataset

# TRAIN TEST SPLIT

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)

print(X_test.shape)


(5616, 35)
(1405, 35)


In [8]:

# FEATURE SCALING

from sklearn.preprocessing import StandardScaler

# Create scaler object
scaler = StandardScaler()

# Fit and transform training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing data
X_test_scaled = scaler.transform(X_test)

print("Scaling Completed")

Scaling Completed


In [9]:
from sklearn.linear_model import LogisticRegression

In [10]:
# Create Logistic Regression model

lr_model = LogisticRegression()
# Train model using scaled data

lr_model.fit(
    X_train_scaled,
    y_train
)

print("Model Training Completed")

# Predict churn values

lr_pred = lr_model.predict(
    X_test_scaled
)

print(lr_pred[:10])

# Probability predictions

lr_prob = lr_model.predict_proba(
    X_test_scaled
)

print(lr_prob[:5])
from sklearn.metrics import accuracy_score

lr_accuracy = accuracy_score(
    y_test,
    lr_pred
)
#ACCURACY
print("Accuracy:", lr_accuracy)
from sklearn.metrics import roc_auc_score

lr_auc = roc_auc_score(
    y_test,
    lr_pred
)

print("ROC-AUC Score:", lr_auc)
# PRECISION
from sklearn.metrics import precision_score

lr_precision = precision_score(
    y_test,
    lr_pred
)

print("Precision:", lr_precision)

#RECALL
from sklearn.metrics import recall_score

lr_recall = recall_score(
    y_test,
    lr_pred
)

print("Recall:", lr_recall)
# CONFUSION MATRIX
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    lr_pred
)

print(cm)
# CLASSIFICATION REPORT
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        lr_pred
    )
)




Model Training Completed
[ True False False False False False False False False False]
[[0.4505395  0.5494605 ]
 [0.70863045 0.29136955]
 [0.92775159 0.07224841]
 [0.99317873 0.00682127]
 [0.55262634 0.44737366]]
Accuracy: 0.79288256227758
ROC-AUC Score: 0.6944484913234913
Precision: 0.6055363321799307
Recall: 0.4971590909090909
[[939 114]
 [177 175]]
              precision    recall  f1-score   support

       False       0.84      0.89      0.87      1053
        True       0.61      0.50      0.55       352

    accuracy                           0.79      1405
   macro avg       0.72      0.69      0.71      1405
weighted avg       0.78      0.79      0.79      1405



In [11]:

# RANDOM FOREST - CUSTOMER CHURN PREDICTION


from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# Create Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train model

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest Training Completed")

# Make predictions

rf_pred = rf_model.predict(
    X_test
)

# Predict probabilities

rf_prob = rf_model.predict_proba(
    X_test
)

# EVALUATION

# Accuracy

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print("\nAccuracy:")
print(rf_accuracy)

# ROC-AUC Score

rf_auc = roc_auc_score(
    y_test,
    rf_pred
)

print("\nROC-AUC Score:")
print(rf_auc)

# Precision

rf_precision = precision_score(
    y_test,
    rf_pred
)

print("\nPrecision:")
print(rf_precision)

# Recall

rf_recall = recall_score(
    y_test,
    rf_pred
)

print("\nRecall:")
print(rf_recall)

# Confusion Matrix

cm = confusion_matrix(
    y_test,
    rf_pred
)

print("\nConfusion Matrix:")
print(cm)

# Classification Report

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        rf_pred
    )
)

# First 10 Predictions

print("\nFirst 10 Predictions:")
print(rf_pred[:10])

# First 5 Probability Predictions

print("\nFirst 5 Probability Predictions:")
print(rf_prob[:5])

# FEATURE IMPORTANCE


importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nTop 10 Important Features:")

print(importance.head(10))

Random Forest Training Completed

Accuracy:
0.7864768683274022

ROC-AUC Score:
0.6778819174652508

Precision:
0.5955882352941176

Recall:
0.4602272727272727

Confusion Matrix:
[[943 110]
 [190 162]]

Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.90      0.86      1053
        True       0.60      0.46      0.52       352

    accuracy                           0.79      1405
   macro avg       0.71      0.68      0.69      1405
weighted avg       0.77      0.79      0.78      1405


First 10 Predictions:
[ True False False False False False False False False False]

First 5 Probability Predictions:
[[0.48 0.52]
 [0.74 0.26]
 [0.72 0.28]
 [0.99 0.01]
 [0.76 0.24]]

Top 10 Important Features:
                           Feature  Importance
3                     TotalCharges    0.142465
1                           tenure    0.136576
2                   MonthlyCharges    0.125147
4                  AvgMonthlySpend    0.124719
5 

In [12]:

# XGBOOST - CUSTOMER CHURN PREDICTION

# Install XGBoost
!pip install xgboost

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# Create XGBoost model

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train model

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost Training Completed")

# Make predictions

xgb_pred = xgb_model.predict(
    X_test
)

# Predict probabilities

xgb_prob = xgb_model.predict_proba(
    X_test
)


# EVALUATION

# Accuracy

xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

print("\nAccuracy:")
print(xgb_accuracy)

# ROC-AUC Score

xgb_auc = roc_auc_score(
    y_test,
    xgb_pred
)

print("\nROC-AUC Score:")
print(xgb_auc)

# Precision

xgb_precision = precision_score(
    y_test,
    xgb_pred
)

print("\nPrecision:")
print(xgb_precision)

# Recall

xgb_recall = recall_score(
    y_test,
    xgb_pred
)

print("\nRecall:")
print(xgb_recall)

# Confusion Matrix

cm = confusion_matrix(
    y_test,
    xgb_pred
)

print("\nConfusion Matrix:")
print(cm)

# Classification Report

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        xgb_pred
    )
)

# First 10 Predictions

print("\nFirst 10 Predictions:")
print(xgb_pred[:10])

# First 5 Probability Predictions

print("\nFirst 5 Probability Predictions:")
print(xgb_prob[:5])


# FEATURE IMPORTANCE


importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nTop 10 Important Features:")

print(importance.head(10))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [05:11:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Training Completed

Accuracy:
0.800711743772242

ROC-AUC Score:
0.7072366291116292

Precision:
0.6224489795918368

Recall:
0.5198863636363636

Confusion Matrix:
[[942 111]
 [169 183]]

Classification Report:
              precision    recall  f1-score   support

       False       0.85      0.89      0.87      1053
        True       0.62      0.52      0.57       352

    accuracy                           0.80      1405
   macro avg       0.74      0.71      0.72      1405
weighted avg       0.79      0.80      0.79      1405


First 10 Predictions:
[1 0 0 0 0 0 0 0 0 0]

First 5 Probability Predictions:
[[0.4533999  0.5466001 ]
 [0.6666707  0.3333293 ]
 [0.80891836 0.19108161]
 [0.98082864 0.01917135]
 [0.5599468  0.44005322]]

Top 10 Important Features:
                           Feature  Importance
12     InternetService_Fiber optic    0.224268
27               Contract_Two year    0.171493
26               Contract_One year    0.085047
13              InternetService_No  